# Deep Agent 统一课程：从 Python 到 LangChain、LangGraph、Deep Agents

> 这是一份按课堂节奏编写的 Notebook。老师会先提出问题，再写最小代码，解释为什么需要下一个抽象，最后让你自己练习。

我们最终要做出的不是一个只能聊天的 Demo，而是一个能够**规划、调用工具、保存状态、失败重试、请求确认、验证结果并持续运行**的 Agent 系统。

## 0. 课程地图：为什么要这样学习？

假设我们要做一个“学习研究助手”：用户提出问题，助手查资料、整理答案、检查引用，必要时请另一个 Agent 审核。这个任务会自然地遇到下面这些问题：

```text
普通 Python       解决确定性逻辑
LangChain         连接模型、Prompt、工具和结构化输出
LangGraph         管理多步骤状态、分支、循环、暂停和恢复
Deep Agents       为长任务预置规划、文件系统、上下文和子 Agent
生产化工程        处理权限、日志、评测、成本和部署
```

学习顺序不是为了背框架，而是为了让每一次升级都解决一个真实问题：先看见问题，再引入抽象。

### 学完本课程，你应该能回答

1. Tool、State、Node、Edge、Checkpointer 分别是什么？
2. 为什么一次模型调用不等于 Agent？
3. 什么时候使用 LangChain Agent，什么时候使用 LangGraph？
4. Deep Agents 帮我们省掉了哪些基础设施？
5. 如何防止 Agent 无限循环、越权访问和输出不可用？
6. 如何用真实任务评估 Agent，而不是只看一两个漂亮回答？

## 1. 环境准备

建议使用 Python 3.11 或更高版本。本机已准备 Python 3.14 环境。模型部分统一使用 DeepSeek 的 OpenAI 兼容接口：模型名为 `deepseek-v4-flash`，接口地址为 `https://api.deepseek.com`。没有 API Key 也可以完成大部分流程图和工程示例；只有模型调用单元格会被跳过。

### 安全配置 API Key

请在 VS Code 终端或项目根目录的 `.env` 文件中设置 `DEEPSEEK_API_KEY`，不要把 Key 直接写进 Notebook，也不要提交到 Git：

```bash
export DEEPSEEK_API_KEY="你的 DeepSeek Key"
```

如果 Key 曾经公开粘贴过，建议去 DeepSeek 控制台撤销并重新生成。

In [27]:
# 第一次运行时执行。Notebook 中使用 %pip，确保依赖安装到当前内核。
%pip install -U "langchain>=1.0" "langgraph>=0.6" "deepagents>=0.7" langchain-openai pydantic python-dotenv jupyter ipykernel

Note: you may need to restart the kernel to use updated packages.


In [2]:
import logging
import operator
import os
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError
from pathlib import Path
from typing import Annotated, TypedDict

from dotenv import load_dotenv
load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash")
RUN_LLM = bool(DEEPSEEK_API_KEY)

if RUN_LLM:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model=MODEL,
        api_key=DEEPSEEK_API_KEY,
        base_url=DEEPSEEK_BASE_URL,
        temperature=0,
    )

print("模型调用：", "启用" if RUN_LLM else "跳过，请配置 DEEPSEEK_API_KEY")
print("模型：", MODEL)
print("接口：", DEEPSEEK_BASE_URL)

模型调用： 启用
模型： deepseek-v4-flash
接口： https://api.deepseek.com


### 老师的学习约定

每个章节都按四步走：

1. **先观察**：先运行最小例子，看输入和输出。
2. **再解释**：明确这一段代码解决了什么问题。
3. **再升级**：只有当前写法开始吃力时，才引入框架能力。
4. **再练习**：改一个参数或增加一个分支，确认自己真的理解。

不要急着跳到最后的 Deep Agents。能够解释中间每一层，遇到问题时才知道该在哪里修。

## 2. 第一课：先不用 LLM，写一个普通程序

老师先给你一个确定性任务：根据关键词从笔记中找答案。这里完全不需要模型。

为什么要从这里开始？因为 Agent 不是“神奇的聊天框”，它仍然需要普通代码负责数据、状态和边界。

In [4]:
notes = {
    "langchain": "LangChain 提供模型、消息、Prompt 和工具抽象。",
    "langgraph": "LangGraph 用节点和边编排有状态流程。",
    "deep agents": "Deep Agents 为长任务预置规划、文件系统、上下文和子 Agent。",
}

def lookup_note(keyword: str) -> str:
    return notes.get(keyword.lower(), "没有找到对应笔记。")

print(lookup_note("langgraph"))
print(lookup_note("unknown"))

LangGraph 用节点和边编排有状态流程。
没有找到对应笔记。


### 老师讲解

这个函数有三个特点：输入明确、输出明确、行为确定。它适合直接写成函数，不需要 Agent。

但它也有局限：用户不会总是输入精确的键；问题可能需要先搜索、再计算、再总结；下一步可能取决于上一步结果。**不确定性**出现以后，我们才开始需要模型和流程编排。

## 3. 第二课：第一次调用模型，只做一件事

先不要做 Agent。我们让模型把一段资料改写成适合初学者的解释。

这一课的重点是理解：模型调用就是输入消息、得到输出消息。后面所有 Agent 框架，最终都要回到这个基本动作。

In [5]:
from langchain_core.prompts import ChatPromptTemplate

teacher_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一位耐心的老师。用短句解释，不使用没有解释的术语。"),
    ("human", "请根据资料回答问题。\n资料：{context}\n问题：{question}"),
])

example_input = {
    "context": notes["langgraph"],
    "question": "LangGraph 解决什么问题？",
}
print(teacher_prompt.invoke(example_input).to_string())

if RUN_LLM:
    response = (teacher_prompt | llm).invoke(example_input)
    print("\n模型回答：", response.content)
else:
    print("\n未设置 API Key：先观察 Prompt 产生的消息。")

System: 你是一位耐心的老师。用短句解释，不使用没有解释的术语。
Human: 请根据资料回答问题。
资料：LangGraph 用节点和边编排有状态流程。
问题：LangGraph 解决什么问题？

模型回答： LangGraph 帮助你把多个步骤组织成一个流程。  
它还让流程记住之前的信息（状态）。  
所以它解决“如何有序管理复杂流程”的问题。


### 老师讲解：为什么要用 PromptTemplate？

把 Prompt 写成模板有三个好处：

- 角色说明和用户问题分开，容易维护。
- 资料是变量，可以被检索结果替换。
- 同一个模板可以反复测试，不必手动拼接字符串。

现在请你做一个小练习：把 `system` 改成“你是一位严格的代码审查老师”，观察回答风格有什么变化。

## 4. 第三课：让模型能够行动，认识 Tool

模型本身只能生成文本。它不能真的查询数据库、读取文件或计算结果。我们把这些能力包装成 Tool，再把 Tool 的说明交给模型。

In [6]:
def calculate(expression: str) -> str:
    """只处理形如“数字 运算符 数字”的简单表达式。"""
    left, op, right = expression.split()
    a, b = float(left), float(right)
    operations = {"+": a + b, "-": a - b, "*": a * b, "/": a / b}
    value = operations[op]
    return str(int(value) if value.is_integer() else value)

print(calculate("12 * 7"))

84


普通函数已经能完成动作。接下来需要补充两类元数据：

1. 工具叫什么。
2. 工具什么时候应该被调用、参数应该怎么填。

LangChain 的 `@tool` 装饰器会把函数转换成模型可以理解的工具描述。工具的 docstring 不是装饰品，它会影响模型的选择。

In [7]:
from langchain_core.tools import tool

@tool
def safe_calculator(expression: str) -> str:
    """计算简单算式，例如 12 * 7。参数必须包含空格。"""
    try:
        return calculate(expression)
    except (ValueError, KeyError, ZeroDivisionError) as exc:
        return f"计算失败，请检查格式：{exc}"

print("工具名：", safe_calculator.name)
print("工具描述：", safe_calculator.description)
print("工具调用：", safe_calculator.invoke("12 * 7"))

工具名： safe_calculator
工具描述： 计算简单算式，例如 12 * 7。参数必须包含空格。
工具调用： 84


## 5. 第四课：ReAct，让模型决定下一步

如果用户问“计算 12 * 7，并解释 LangGraph”，程序需要先判断：应该调用计算工具，也应该查询笔记。这个“观察 -> 选择行动 -> 读取结果 -> 再决定”的循环，就是 ReAct 的核心。

LangChain 的 `create_agent` 已经帮我们实现了基础循环。你需要关注的是：工具描述是否清楚，模型是否拿到了工具结果，以及最终结果是否需要验证。

In [8]:
@tool
def search_learning_notes(query: str) -> str:
    """根据关键词搜索 LangChain、LangGraph、Deep Agents 学习笔记。"""
    hits = [text for key, text in notes.items() if query.lower() in key or query.lower() in text.lower()]
    return "\n".join(hits) if hits else "没有找到相关笔记。"

if RUN_LLM:
    from langchain.agents import create_agent
    react_agent = create_agent(
        model=llm,
        tools=[safe_calculator, search_learning_notes],
        system_prompt="你是学习助手。需要计算时使用计算工具，需要资料时使用搜索工具。不要编造工具没有返回的事实。",
    )
    react_result = react_agent.invoke({"messages": [{"role": "user", "content": "计算 12 * 7，并解释 LangGraph。"}]})
    print(react_result["messages"][-1].content)
else:
    print("未设置 API Key：ReAct 模型示例跳过。上面的 Tool 仍然可以单独调用。")

两个问题都有结果了。

## 1. 计算结果

**12 × 7 = 84**

## 2. 关于 LangGraph

我搜索了学习笔记，找到的信息比较简短，原文如下：

> 「LangGraph 用节点和边编排有状态流程。」

基于这条笔记可以这样理解：

- **节点（Node）**：表示流程中的一个个处理步骤，比如一个 LLM 调用、一次工具执行或一个条件判断。
- **边（Edge）**：定义节点之间的流转路径，即前一步完成后下一步该去往哪里。
- **有状态（Stateful）**：流程会维护并传递一个共享状态，各个节点可以读写这个状态，从而让多步骤任务能够连贯地协同工作。

需要说明的是，笔记库中目前只有这一条简短记录，如果你想深入了解 LangGraph 的具体用法（比如状态定义、条件边、如何构建图），我可以进一步帮你查找相关资料或梳理一个示例。


### 检查点 1：你现在应该理解什么？

- 普通函数是真正执行动作的地方。
- Tool 是带有名称、描述和参数契约的函数。
- Agent 负责决定调用哪个 Tool，以及是否继续。
- ReAct 适合工具选择不确定、步骤数量不固定的任务。

如果流程是固定的，就不必强行使用 Agent。下一课我们先用确定的 LangGraph 流程理解 State。

## 6. 第五课：LangGraph 的核心是 State

先想一个具体问题：研究助手执行到一半时，系统需要知道什么？

至少需要知道：用户问题、已经找到的事实、当前草稿、是否通过检查。我们把这些信息定义成 State。

In [10]:
from langgraph.graph import END, START, StateGraph

class ResearchState(TypedDict):
    question: str
    facts: list[str]
    draft: str
    checked: bool

def collect_facts(state: ResearchState):
    return {"facts": ["LangGraph 用 StateGraph 表达流程。", "节点函数读取并更新 State。"]}

def write_draft(state: ResearchState):
    return {"draft": "；".join(state["facts"]) }

def check_draft(state: ResearchState):
    return {"checked": len(state["draft"]) > 10}

builder = StateGraph(ResearchState)
builder.add_node("collect", collect_facts)
builder.add_node("draft", write_draft)
builder.add_node("check", check_draft)
builder.add_edge(START, "collect")
builder.add_edge("collect", "draft")
builder.add_edge("draft", "check")
builder.add_edge("check", END)
research_graph = builder.compile()

result = research_graph.invoke({
    "question": "LangGraph 是什么？",
    "facts": [], "draft": "", "checked": False,
})
print(result)

{'question': 'LangGraph 是什么？', 'facts': ['LangGraph 用 StateGraph 表达流程。', '节点函数读取并更新 State。'], 'draft': 'LangGraph 用 StateGraph 表达流程。；节点函数读取并更新 State。', 'checked': True}


### 老师逐行讲解

- `ResearchState` 是任务快照，规定了允许保存哪些数据。
- `add_node` 把一个普通 Python 函数注册成节点。
- `add_edge` 规定节点的先后关系。
- `compile` 把定义变成可执行图。
- `invoke` 传入初始 State，最后拿到完整 State。

注意：节点不需要返回完整 State，只返回自己负责修改的字段。LangGraph 会把更新合并到当前状态中。

## 7. 第六课：条件边和循环，让系统能够修正自己

刚才的流程无论草稿好坏都会结束。真实系统需要判断：检查不通过就重写，通过才结束。

这就是 LangGraph 的条件边。为了防止无限循环，我们同时记录 `attempts`。

In [11]:
class ReviewState(TypedDict):
    draft: str
    attempts: int
    score: int
    feedback: str

def write_review_draft(state: ReviewState):
    attempts = state["attempts"] + 1
    draft = (
        "LangGraph 负责流程。"
        if attempts == 1
        else "LangChain 提供工具；LangGraph 管理状态；Deep Agents 支持长任务。"
    )
    return {"draft": draft, "attempts": attempts}

def review_draft(state: ReviewState):
    required = ["LangChain", "LangGraph", "Deep Agents"]
    score = sum(word in state["draft"] for word in required)
    return {"score": score, "feedback": "通过" if score == 3 else "信息不完整"}

def route_review(state: ReviewState):
    if state["score"] >= 3 or state["attempts"] >= 3:
        return "finish"
    return "retry"

review_builder = StateGraph(ReviewState)
review_builder.add_node("write", write_review_draft)
review_builder.add_node("review", review_draft)
review_builder.add_edge(START, "write")
review_builder.add_edge("write", "review")
review_builder.add_conditional_edges("review", route_review, {"retry": "write", "finish": END})
review_graph = review_builder.compile()

review_result = review_graph.invoke({"draft": "", "attempts": 0, "score": 0, "feedback": ""})
print(review_result)

{'draft': 'LangChain 提供工具；LangGraph 管理状态；Deep Agents 支持长任务。', 'attempts': 2, 'score': 3, 'feedback': '通过'}


## 8. 第七课：Planner-Executor，把想法和动作分开

当任务变长时，边想边做容易漏步骤。我们把角色拆成：

- **Planner**：根据目标产生计划。
- **Executor**：一次执行一个步骤。
- **Router**：判断还有没有下一步。

这是很多复杂 Agent 的基本骨架。

In [12]:
class PlanState(TypedDict):
    goal: str
    plan: list[str]
    current: int
    results: list[str]

def planner(state: PlanState):
    return {"plan": ["明确问题", "查找资料", "验证答案"], "current": 0}

def executor(state: PlanState):
    index = state["current"]
    step = state["plan"][index]
    return {"current": index + 1, "results": state["results"] + [f"完成：{step}"]}

def route_plan(state: PlanState):
    return "next" if state["current"] < len(state["plan"]) else "finish"

plan_builder = StateGraph(PlanState)
plan_builder.add_node("planner", planner)
plan_builder.add_node("executor", executor)
plan_builder.add_edge(START, "planner")
plan_builder.add_edge("planner", "executor")
plan_builder.add_conditional_edges("executor", route_plan, {"next": "executor", "finish": END})
plan_graph = plan_builder.compile()
plan_result = plan_graph.invoke({"goal": "研究 Agent", "plan": [], "current": 0, "results": []})
print("计划：", plan_result["plan"])
print("结果：", plan_result["results"])

计划： ['明确问题', '查找资料', '验证答案']
结果： ['完成：明确问题', '完成：查找资料', '完成：验证答案']


## 9. 第八课：Memory 和 Checkpointer

如果 Notebook 中断，或者同一个用户第二次提问，Agent 如何知道之前发生了什么？答案是保存 State。

LangGraph 用 `checkpointer` 按 `thread_id` 保存状态。这里先使用内存版，生产环境再替换成数据库实现。

In [13]:
from langgraph.checkpoint.memory import InMemorySaver

class ChatState(TypedDict):
    messages: Annotated[list[str], operator.add]

def remember_message(state: ChatState):
    latest = state["messages"][-1]
    return {"messages": [f"系统记录：{latest}"]}

memory_builder = StateGraph(ChatState)
memory_builder.add_node("remember", remember_message)
memory_builder.add_edge(START, "remember")
memory_builder.add_edge("remember", END)
memory_graph = memory_builder.compile(checkpointer=InMemorySaver())

thread = {"configurable": {"thread_id": "student-001"}}
memory_graph.invoke({"messages": ["我正在学习 LangGraph"]}, thread)
memory_graph.invoke({"messages": ["下一步学习 Deep Agents"]}, thread)
print(memory_graph.get_state(thread).values)

{'messages': ['我正在学习 LangGraph', '系统记录：我正在学习 LangGraph', '下一步学习 Deep Agents', '系统记录：下一步学习 Deep Agents']}


## 10. 第九课：上下文不是越多越好

长任务会产生大量工具输出。如果每次都把全部历史塞给模型，会带来三个问题：上下文超限、成本上涨、模型注意力下降。

常见策略是：保留早期摘要、保留最近消息、把大结果放到外部文件，并在 State 中只保存文件路径。

In [14]:
def compress_messages(messages: list[str], max_chars: int = 120) -> str:
    text = "\n".join(messages)
    if len(text) <= max_chars:
        return text
    summary = messages[0][:45]
    recent = "\n".join(messages[-2:])
    return f"[早期摘要] {summary}...\n[最近消息]\n{recent}"

history = [
    "用户想了解 Agent 的状态、工具、规划和持久化。",
    "检索工具返回了很多文档片段。",
    "用户要求例子简单，并且需要一步步讲解。",
    "下一步是实现一个小型学习助手。",
]
print(compress_messages(history))

用户想了解 Agent 的状态、工具、规划和持久化。
检索工具返回了很多文档片段。
用户要求例子简单，并且需要一步步讲解。
下一步是实现一个小型学习助手。


## 11. 第十课：失败是正常路径，不是例外

网络请求会超时，API 会限流，工具参数会错误。可靠 Agent 不会假装一切都成功，而是把失败设计进流程。

老师建议先区分三种情况：

- 临时错误：可以有限重试。
- 可修复错误：修改参数或回到上一步。
- 高风险错误：停止并请求人工处理。

In [15]:
def call_with_retry(fn, max_attempts: int = 3):
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn(), attempt
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"超过最大重试次数：{last_error}")

counter = {"value": 0}
def temporary_failure():
    counter["value"] += 1
    if counter["value"] < 3:
        raise ConnectionError("模拟网络抖动")
    return "成功"

print(call_with_retry(temporary_failure))

def slow_tool():
    time.sleep(0.15)
    return "完成"

with ThreadPoolExecutor(max_workers=1) as pool:
    future = pool.submit(slow_tool)
    try:
        print(future.result(timeout=0.01))
    except TimeoutError:
        print("超时：可以取消、重试或转人工。")

('成功', 3)
超时：可以取消、重试或转人工。


## 12. 第十一课：高风险操作必须有人确认

研究助手可以自动整理摘要，但“发布文章”“删除文件”“发送邮件”应该在动作前暂停。

LangGraph 的 `interrupt` 会把待确认信息返回给用户，用户决定后再通过 `Command(resume=...)` 恢复图。

In [16]:
from langgraph.types import Command, interrupt

class ApprovalState(TypedDict):
    action: str
    approved: bool
    result: str

def ask_approval(state: ApprovalState):
    decision = interrupt({"question": "是否执行动作？", "action": state["action"]})
    return {"approved": bool(decision)}

def apply_approval(state: ApprovalState):
    return {"result": "动作已执行" if state["approved"] else "动作被拒绝"}

approval_builder = StateGraph(ApprovalState)
approval_builder.add_node("approval", ask_approval)
approval_builder.add_node("apply", apply_approval)
approval_builder.add_edge(START, "approval")
approval_builder.add_edge("approval", "apply")
approval_builder.add_edge("apply", END)
approval_graph = approval_builder.compile(checkpointer=InMemorySaver())
approval_config = {"configurable": {"thread_id": "approval-001"}}
paused = approval_graph.invoke({"action": "发布 v1.0", "approved": False, "result": ""}, approval_config)
print("暂停信息：", paused.get("__interrupt__"))
resumed = approval_graph.invoke(Command(resume=True), approval_config)
# resumed = approval_graph.invoke(Command(resume=False), approval_config)
print("恢复结果：", resumed["result"])

暂停信息： [Interrupt(value={'question': '是否执行动作？', 'action': '发布 v1.0'}, id='cb817dbbecaaf20d7cfd7bfe2422b7d3')]
恢复结果： 动作已执行


## 13. 第十二课：结果必须满足契约

自然语言适合给人阅读，但不适合直接交给下一个程序。比如前端需要答案、置信度和来源，就应该定义结构，而不是靠字符串解析。

In [17]:
from pydantic import BaseModel, Field

class StudyAnswer(BaseModel):
    answer: str = Field(description="简短答案")
    confidence: int = Field(ge=0, le=100, description="0 到 100")
    sources: list[str] = Field(default_factory=list)

candidate = {"answer": "State 是任务的结构化快照。", "confidence": 95, "sources": ["note-1"]}
validated = StudyAnswer.model_validate(candidate)
print(validated.model_dump())

if RUN_LLM:
    structured_llm = llm.with_structured_output(StudyAnswer)
    answer = structured_llm.invoke("解释 State，并返回置信度和来源。")
    print(answer.model_dump())
else:
    print("未设置 API Key：只演示 Pydantic 验证。")

{'answer': 'State 是任务的结构化快照。', 'confidence': 95, 'sources': ['note-1']}


BadRequestError: Error code: 400 - {'error': {'message': 'This response_format type is unavailable now', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

## 14. 第十三课：RAG，从检索开始

RAG 不是“把所有文档塞给模型”，而是：

```text
问题 -> 检索相关片段 -> 组成上下文 -> 模型回答 -> 返回来源
```

先用最简单的关键词检索理解流程，之后再把它替换成 Embedding 和向量数据库。

In [18]:
from langchain_core.documents import Document

knowledge_base = [
    Document(page_content="LangChain 提供模型、消息、Prompt 和工具抽象。", metadata={"source": "langchain.md"}),
    Document(page_content="LangGraph 用 StateGraph、节点和边管理有状态流程。", metadata={"source": "langgraph.md"}),
    Document(page_content="Deep Agents 预置规划、文件系统、上下文管理和子 Agent。", metadata={"source": "deepagents.md"}),
]

def retrieve_documents(query: str, k: int = 2) -> list[Document]:
    keywords = [word.lower() for word in query.split() if word.strip()]
    scored = []
    for document in knowledge_base:
        text = document.page_content.lower()
        score = sum(keyword in text for keyword in keywords)
        scored.append((score, document))
    return [doc for score, doc in sorted(scored, key=lambda item: item[0], reverse=True)[:k] if score > 0]

hits = retrieve_documents("LangGraph 状态流程")
context = "\n".join(doc.page_content for doc in hits)
print("检索结果：", [(doc.metadata["source"], doc.page_content) for doc in hits])

if RUN_LLM:
    rag_answer = (teacher_prompt | llm).invoke({"context": context, "question": "LangGraph 解决什么问题？"})
    print("回答：", rag_answer.content)
else:
    print("未设置 API Key：检索已完成，模型回答跳过。")

检索结果： [('langgraph.md', 'LangGraph 用 StateGraph、节点和边管理有状态流程。')]
回答： LangGraph 帮助管理复杂流程。  
这种流程有很多步骤。  
每一步需要记住之前的信息。  
这叫“有状态流程”。  

StateGraph 就是一张图。  
它表示整个流程的结构。  
节点是图里的一个步骤。  
边是步骤之间的连线。  
它们告诉你怎么从一个步骤走到下一步。  

LangGraph 解决的核心问题是：  
如何清楚地安排多个步骤，  
并且让每一步都能使用前面步骤留下的信息。


## 15. 第十四课：Deep Agents，为什么需要更高层的 Harness？

到目前为止，我们已经手动实现了很多能力：计划、工具、状态、循环、记忆、重试。长任务继续发展时，还会需要文件系统、上下文卸载、子 Agent 和技能管理。

Deep Agents 的定位是：在 LangGraph 运行时之上，提供一套适合长周期、多步骤任务的默认组合。它不是替代 LangChain 或 LangGraph，而是复用它们并减少样板代码。

In [19]:
from deepagents import create_deep_agent

if RUN_LLM:
    deep_agent = create_deep_agent(
        model=llm,
        tools=[safe_calculator, search_learning_notes],
        system_prompt=(
            "你是学习研究助手。面对多步骤任务时先形成计划，"
            "需要资料就使用搜索工具，最后给出简短且可验证的答案。"
        ),
    )
    deep_result = deep_agent.invoke({"messages": [{"role": "user", "content": "研究 LangGraph 和 Deep Agents 的区别，并引用学习笔记。"}]})
    print(deep_result["messages"][-1].content)
else:
    print("未设置 API Key：Deep Agents 模型示例跳过。")

我搜索了学习笔记库，仅有两篇相关笔记可用作引用（其余关键词无命中）。以下是结论，引文均已标注。

## 学习笔记引文

- **LangGraph**：> “用节点和边编排有状态流程。”
- **Deep Agents**：> “为长任务预置规划、文件系统、上下文和子 Agent。”

## 基于笔记的对比

| 维度 | LangGraph（笔记:节点/边/有状态流程） | Deep Agents（笔记:长任务+规划+文件系统+上下文+子 Agent） |
|---|---|---|
| 抽象层次 | **底层编排框架**——把 Agent 逻辑描述成一张状态图 | **高层预设范式**——为“长任务”直接给出整套结构 |
| 流程构建 | 由你定义节点和边，自己拼接出控制流 | 规划、文件系统、上下文等已预置，开箱即用 |
| 状态管理 | “有状态流程”：状态在节点间显式共享与流转 | 通过“上下文”机制管理，配合子 Agent 隔离 |
| 规划能力 | 不强制，需自行在节点里实现（如 ReAct/规划-执行） | “规划”是内置环节，为长任务拆解步骤 |
| 子 Agent | 需手动用子图(subgraph)等方式组装 | 结构上天然支持“子 Agent”层级协作 |
| 文件系统 | 不涉及，需自己接工具 | 内置文件系统能力，供长任务读写中间产物 |

## 一句话总结

- **LangGraph 给你“画笔和画布”**：用节点/边精确编排有状态流程，灵活但需要你自己设计大部分结构（适合定制化、需要细粒度控制的工作流）。
- **Deep Agents 给你“成品模板”**：面向长任务，把规划、文件系统、上下文、子 Agent 都预设好（适合快速构建能自主跑很久的多智能体系统）。

可验证方式：在笔记库中搜索「LangGraph」与「Deep Agents」即可复现上述两条引文。若需要，我可以进一步帮你整理 LangChain 生态中两者的定位差异，或补一篇更完整的分析文档。


### Deep Agents 的子 Agent

主 Agent 不需要亲自做所有事情。研究、代码审查、事实核查可以交给隔离上下文的子 Agent。子 Agent 的描述越具体，主 Agent 越容易在正确的时机委派。

In [20]:
if RUN_LLM:
    subagents = [{
        "name": "reviewer",
        "description": "审核学习摘要是否遗漏 LangChain、LangGraph、Deep Agents 的关键区别。",
        "system_prompt": "你是严谨的审核员。只报告有证据的问题，并给出具体修改建议。",
    }]
    agent_with_reviewer = create_deep_agent(
        model=llm,
        tools=[search_learning_notes],
        subagents=subagents,
        system_prompt="先完成摘要，需要检查时委派 reviewer。",
    )
    reviewed = agent_with_reviewer.invoke({"messages": [{"role": "user", "content": "写摘要并请 reviewer 检查。"}]})
    print(reviewed["messages"][-1].content)
else:
    print("未设置 API Key：子 Agent 示例跳过。")

KeyboardInterrupt: 

## 16. 第十五课：把知识组合成一个小型学习助手

现在我们不再只看单个 API，而是把前面的思想组合起来：

```text
接收问题 -> 检索资料 -> 制定计划 -> 生成答案 -> 验证答案 -> 返回来源
                         |
                    失败则重试
```

下面先用确定性节点实现骨架。等你理解 State 和 Edge 后，再把某个节点替换成真实模型。

In [21]:
class StudyAssistantState(TypedDict):
    question: str
    documents: list[str]
    plan: list[str]
    answer: str
    sources: list[str]
    valid: bool
    attempts: int

def capstone_retrieve(state: StudyAssistantState):
    hits = retrieve_documents(state["question"])
    return {"documents": [doc.page_content for doc in hits], "sources": [doc.metadata["source"] for doc in hits]}

def capstone_plan(state: StudyAssistantState):
    return {"plan": ["阅读相关资料", "组织答案", "检查来源"]}

def capstone_answer(state: StudyAssistantState):
    answer = "；".join(state["documents"])
    return {"answer": answer, "attempts": state["attempts"] + 1}

def capstone_validate(state: StudyAssistantState):
    valid = bool(state["answer"]) and bool(state["sources"])
    return {"valid": valid}

def capstone_route(state: StudyAssistantState):
    return "finish" if state["valid"] or state["attempts"] >= 2 else "retry"

capstone_builder = StateGraph(StudyAssistantState)
capstone_builder.add_node("retrieve", capstone_retrieve)
capstone_builder.add_node("plan", capstone_plan)
capstone_builder.add_node("answer", capstone_answer)
capstone_builder.add_node("validate", capstone_validate)
capstone_builder.add_edge(START, "retrieve")
capstone_builder.add_edge("retrieve", "plan")
capstone_builder.add_edge("plan", "answer")
capstone_builder.add_edge("answer", "validate")
capstone_builder.add_conditional_edges("validate", capstone_route, {"retry": "answer", "finish": END})
capstone_graph = capstone_builder.compile()

capstone_result = capstone_graph.invoke({
    "question": "LangGraph 状态流程", "documents": [], "plan": [],
    "answer": "", "sources": [], "valid": False, "attempts": 0,
})
print(StudyAnswer(answer=capstone_result["answer"], confidence=90, sources=capstone_result["sources"]).model_dump())

{'answer': 'LangGraph 用 StateGraph、节点和边管理有状态流程。', 'confidence': 90, 'sources': ['langgraph.md']}


## 17. 第十六课：评测，不要凭感觉判断 Agent

一个 Demo 回答得好，不代表系统可靠。我们要准备一组真实问题，为每个问题定义最低要求，然后计算通过率。

最简单的评测可以是关键词匹配；更成熟的评测还会检查来源、格式、工具调用和人工评分。

In [ ]:
evaluation_cases = [
    {"question": "LangGraph 是什么？", "must_contain": "状态"},
    {"question": "Deep Agents 有什么能力？", "must_contain": "规划"},
]
predictions = [
    "LangGraph 用 State 管理状态流程。",
    "Deep Agents 提供规划和子 Agent 能力。",
]

checks = [case["must_contain"] in answer for case, answer in zip(evaluation_cases, predictions)]
print({"passed": sum(checks), "total": len(checks), "accuracy": sum(checks) / len(checks)})

## 18. 第十七课：从 Notebook 走向生产

上线前至少回答四个问题：

1. **可观察吗？** 能否知道哪次调用失败、用了哪个工具、花了多少钱？
2. **可控制吗？** 是否限制了文件、命令、网络和高风险动作？
3. **可恢复吗？** 进程重启后能否从 State 继续？
4. **可评估吗？** 新 Prompt 或新模型上线前，能否在固定任务集上回归？

In [ ]:
logger = logging.getLogger("study_agent")
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s %(message)s")

logger.info("agent_started run_id=%s", "run-001")
logger.info("tool_called name=%s", "search_learning_notes")
logger.info("agent_finished run_id=%s", "run-001")

workspace = (Path.cwd() / "agent-workspace").resolve()
def safe_workspace_path(relative_path: str) -> Path:
    target = (workspace / relative_path).resolve()
    if target != workspace and workspace not in target.parents:
        raise PermissionError("超出 Agent 工作目录")
    return target

print(safe_workspace_path("reports/answer.md"))
try:
    safe_workspace_path("../../etc/passwd")
except PermissionError as exc:
    print("安全拦截：", exc)

## 19. 第十八课：调试实验

老师故意给出一个容易失控的路由函数：它只要分数不足就一直重试。请先阅读，再运行修正版。

调试 Agent 时，第一件事不是修改 Prompt，而是检查 State 有没有记录尝试次数、路由是否有终点、工具错误是否被区分。

In [ ]:
def unsafe_route(state: dict) -> str:
    # 如果 score 永远小于 3，这个函数会让图无限循环。
    return "retry" if state["score"] < 3 else "finish"

def safe_route(state: dict, max_attempts: int = 3) -> str:
    # 生产代码必须同时检查质量条件和预算条件。
    if state["score"] >= 3 or state["attempts"] >= max_attempts:
        return "finish"
    return "retry"

print(unsafe_route({"score": 1}))
print(safe_route({"score": 1, "attempts": 3}))

## 20. 四周学习计划：每天知道该练什么

### 第 1 周：Python 和 LangChain

完成普通函数、环境变量、JSON、SQLite；能写一个 Prompt 和两个带类型的 Tool。

### 第 2 周：RAG 和 Agent

完成文档切分、检索、来源返回；配置 API Key，观察 ReAct Agent 的工具调用过程。

### 第 3 周：LangGraph

完成 StateGraph、条件边、循环、Checkpointer、Interrupt；实现可以暂停和恢复的研究任务。

### 第 4 周：Deep Agents 和生产化

使用 Deep Agents 做一个长任务，增加 reviewer 子 Agent；补上日志、评测、权限和成本限制。

每天的学习闭环：读一个概念 -> 跑一个例子 -> 改一个参数 -> 记录一个失败原因。

## 21. 最终项目要求

请自己实现一个“代码仓库学习助手”或“个人知识库研究助手”，至少包含：

- 一个 LangChain Tool。
- 一个 LangGraph StateGraph。
- 一个条件循环和最大尝试次数。
- 一个 Checkpointer，支持按 thread 恢复。
- 一个结构化的最终输出。
- 一个高风险动作的人工确认点。
- 一组至少 10 条评测样本。
- 日志、权限边界和成本限制。

完成这个项目后，再把其中的长任务部分替换成 Deep Agents。这样你会知道 Deep Agents 替你封装了什么，也知道什么时候需要自己回到 LangGraph 定制。

## 22. 术语表

| 术语 | 一句话理解 |
| --- | --- |
| Model | 负责理解和生成的语言模型 |
| Prompt | 给模型的角色、资料和任务说明 |
| Tool | 具有输入输出契约的外部动作 |
| Agent | 能根据结果选择下一步的模型驱动循环 |
| State | 当前任务的结构化快照 |
| Node | 读取和更新 State 的函数 |
| Edge | 节点之间的固定或条件连接 |
| Checkpointer | 保存 State、支持恢复的组件 |
| RAG | 先检索资料，再让模型基于资料回答 |
| Deep Agents | 面向长任务的预置 Agent Harness |

## 23. 继续阅读

- [LangChain Python 文档](https://docs.langchain.com/oss/python/langchain/overview)
- [LangGraph Python 文档](https://docs.langchain.com/oss/python/langgraph/overview)
- [Deep Agents 文档](https://docs.langchain.com/oss/python/deepagents/overview)
- [LangChain Agent 文档](https://docs.langchain.com/oss/python/langchain/agents)

阅读方式：先看概念，再看最小示例，最后把示例改成自己的业务问题。不要只复制 Quickstart；能解释 State 和工具边界，才算真正学会。